In [ ]:
import pandas as pd 
from plotnine import *
import numpy as np
from vpop_calibration import *

%load_ext autoreload
%autoreload 2

In [ ]:
df = pd.read_csv("Simulated_WBC_pacl_ddmore_samePK_nlmixr.csv")

obs_df = (df[df.EVID == 0][["ID", "DV", "V1I", "V2I", "CLI", "TIME"]]
          .rename(columns={"ID": "id", "DV": "value", "V1I": "V1", "V2I": "V2",
                           "CLI": "CL", "TIME": "time"})
          .astype({"V1": float, "V2": float, "CL": float}))

d = df[df.EVID != 0].sort_values(["ID", "TIME"])
starts = d[d.RATE > 0].assign(k=lambda x: x.groupby("ID").cumcount())
stops  = d[d.RATE < 0].assign(k=lambda x: x.groupby("ID").cumcount())

p = starts.merge(stops[["ID", "k", "TIME"]], on=["ID", "k"], suffixes=("", "_end"))
p["dur"] = (p.TIME_end - p.TIME).round().astype(int)

doses = p.pivot(index="ID", columns="k", values=["TIME", "dur", "AMT"]).fillna(0)
doses.columns = [f"{ {'TIME':'time','dur':'occurences','AMT':'dose'}[a] }{b+1}" for a, b in doses.columns]
doses = (doses.reset_index().rename(columns={"ID": "id"})
              [["id", "time1", "occurences1", "dose1", "time2", "occurences2", "dose2"]])

obs_df = obs_df.merge(doses, on="id")
obs_df["value"] = np.log(obs_df["value"])
obs_df["time"] = obs_df["time"] * 3600
obs_df["output_name"] = "CIRC"
obs_df["protocol_arm"] = "identity"
obs_df

In [ ]:
model = SimworkModelBinding(
    path_to_model="CM_Friberg-myelosuppression-model.json",
    path_to_solving_options="SV_Friberg-myelosuppression-model.json",
    inputs=["CIRC0", "MTT", "SLOPU", "GAMMA", "CL", "V1", "V2", "time1","dose1","occurences1","time2","dose2","occurences2"],
    outputs=["CIRC"],
)
print(model.inputs)

struct_model = StructuralSimwork(model=model)

In [ ]:
prior_pdu = {
    "pdu": {
        "CIRC0": {"prior": 7.21, "prior_omega": 0.1},
        "MTT": {"prior": 124, "prior_omega": 0.03},
        "SLOPU": {"prior": 28.9, "prior_omega": 0.2},
        "GAMMA": {"prior": 0.239, "prior_omega": 0.01},
    },
    "pdk": {"time1","time2","dose1","dose2","occurences1", "occurences2", "CL", "V1", "V2"},
    "error_model": {
        "CIRC": {"error_type": "additive", "sigma": 0.5},
    },
}

config = Config(
    saem=SaemConfigDict(
        nb_iter_burnin=0,
        nb_iter_learning=100,
        nb_iter_smoothing=100,
        plot_frames=5,
        optim_max_fun=20,
    ),
    nlme=NlmeConfigDict(nb_chains=1),
)

nlme_model = NlmeModel(
    df=obs_df, prior_params=prior_pdu, structural_model=struct_model, config=config
)

In [ ]:
nlme_model.optimizer.run()

In [ ]:
nlme_model.diagnostics.sample_conditional_distribution(200)

In [ ]:
nlme_model.plot.map_estimates_gof()

In [ ]:
nlme_model.plot.map_estimates()

In [ ]:
nlme_model.plot.weighted_residuals("iwres")

In [ ]:
nlme_model.plot.weighted_residuals("pwres")

In [ ]:
nlme_model.plot.vpc()

In [ ]:
nlme_model.plot.conditional_codistributions()